# Comparación de clasificadores — dataset DNU

Construye un dataset unificado con los resultados de cuatro fuentes de clasificación:

| Fuente | Archivo | Cobertura |
|--------|---------|----------|
| Labels manuales (base) | `tweets_dnu_clasificados.csv` | 259 752 tweets |
| GPT (batch API) | `DNU_gpt_xenofobia.csv` | 259 752 tweets |
| Modelos HuggingFace | `DNU_modelos_hf.csv` | 259 752 tweets |
| pysentimiento | `DNU_pysentiment.csv` | 58 534 tweets (solo 2021-03-05) |

Los cuatro archivos tienen las filas en el mismo orden, y pysentimiento corresponde exactamente a las primeras 58 534 filas. La unificación es **puramente posicional** (sin merge por ID), lo que evita duplicados.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## 2. Carga de datos

In [ ]:
# ── Columnas a conservar de cada archivo ─────────────────────────────────
COLS_BASE = [
    'id', 'text', 'text_pp',
    'fecha', 'fecha_dia', 'is_retweet',
    'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS',
    'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL',
    'labels', 'n_labels',
]

COLS_GPT = ['batch_label', 'batch_is_xenophobic']

COLS_HF = [
    'cardiff_xlm_hate_es__raw_label',  'cardiff_xlm_hate_es__score',  'cardiff_xlm_hate_es__binary_pred',
    'bne_hate_offensive_es__raw_label', 'bne_hate_offensive_es__score', 'bne_hate_offensive_es__binary_pred',
    'beto_hate_v2_es__raw_label',       'beto_hate_v2_es__score',       'beto_hate_v2_es__binary_pred',
]

COLS_PYS = [
    'pysentimiento_hate__raw_label', 'pysentimiento_hate__score',
    'pysentimiento_hate__binary_pred', 'pysentimiento_hate__raw_probas',
]

In [ ]:
df_base = pd.read_csv('data/tweets_dnu_clasificados.csv', usecols=COLS_BASE)
df_gpt  = pd.read_csv('data/DNU_gpt_xenofobia.csv',       usecols=COLS_GPT)
df_hf   = pd.read_csv('data/DNU_modelos_hf.csv',          usecols=COLS_HF)
df_pys  = pd.read_csv('data/DNU_pysentiment.csv',         usecols=COLS_PYS)

print(f"Base:           {df_base.shape}")
print(f"GPT:            {df_gpt.shape}")
print(f"HuggingFace:    {df_hf.shape}")
print(f"pysentimiento:  {df_pys.shape}  ← primeras {len(df_pys):,} filas (solo 2021-03-05)")

## 3. Unificación

**Estrategia posicional:**
Los cuatro archivos comparten el mismo orden de filas. Se hace `pd.concat` por columnas (`axis=1`).  
Para pysentimiento, se reindexan sus columnas al rango completo (los 201 218 tweets que no están cubiertos quedan como `NaN`).

> **Nota:** el dataset base tiene 135 IDs duplicados (0.05 %); se conservan.

In [ ]:
# Renombrar columnas GPT
df_gpt = df_gpt.rename(columns={
    'batch_label':         'gpt_label',
    'batch_is_xenophobic': 'gpt_is_xenophobic',
})

# Extender pysentimiento al tamaño total (NaN para las filas no cubiertas)
n_total = len(df_base)
df_pys_full = df_pys.reindex(range(n_total))  # filas > 58533 quedan NaN

# Concatenación horizontal
df = pd.concat([df_base, df_gpt, df_hf, df_pys_full], axis=1)

print(f"Dataset unificado: {df.shape}")
print(f"\nColumnas ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")

## 4. Descripción del dataset unificado

### 4.1 Cobertura por clasificador

In [ ]:
CLASIFICADORES = {
    'Labels manuales — RACISM':   'RACISM',
    'GPT':                        'gpt_label',
    'Cardiff XLM-RoBERTa (HF)':   'cardiff_xlm_hate_es__binary_pred',
    'BNE Hate Offensive (HF)':    'bne_hate_offensive_es__binary_pred',
    'BETO Hate v2 (HF)':          'beto_hate_v2_es__binary_pred',
    'pysentimiento':              'pysentimiento_hate__binary_pred',
}

resumen = []
for nombre, col in CLASIFICADORES.items():
    n = df[col].notna().sum()
    resumen.append({
        'Clasificador':      nombre,
        'Columna (binary)':  col,
        'Tweets cubiertos':  n,
        '% cobertura':       f"{n / n_total * 100:.1f}%",
    })

pd.DataFrame(resumen).set_index('Clasificador')

### 4.2 Distribución de predicciones

In [ ]:
# Labels manuales
LABEL_COLS = ['CALLS','WOMEN','LGBTI','RACISM','CLASS','POLITICS','DISABLED','APPEARANCE','CRIMINAL']
print("── Labels manuales (tweets con label=True) ──")
display(df[LABEL_COLS].sum().sort_values(ascending=False).rename('conteo').to_frame())

# GPT
print("\n── GPT label ──")
display(df['gpt_label'].value_counts(dropna=False).rename('conteo').to_frame())

# HF models
print("\n── Predicciones binarias — modelos HuggingFace ──")
hf_pred_cols = [
    'cardiff_xlm_hate_es__binary_pred',
    'bne_hate_offensive_es__binary_pred',
    'beto_hate_v2_es__binary_pred',
]
display(
    df[hf_pred_cols].apply(lambda s: s.value_counts(dropna=False))
    .T.rename(columns={0: 'no_hate (0)', 1: 'hate (1)'})
)

# pysentimiento (solo sobre su subconjunto)
print("\n── pysentimiento (sobre las primeras 58 534 filas) ──")
display(df['pysentimiento_hate__binary_pred'].value_counts(dropna=False).rename('conteo').to_frame())

### 4.3 Vista previa del dataset

In [ ]:
COLS_PREVIEW = [
    'id', 'text',
    'RACISM', 'n_labels',
    'gpt_label', 'gpt_is_xenophobic',
    'cardiff_xlm_hate_es__binary_pred',
    'bne_hate_offensive_es__binary_pred',
    'beto_hate_v2_es__binary_pred',
    'pysentimiento_hate__binary_pred',
]
df[COLS_PREVIEW].head(10)

## 5. Guardar dataset unificado

In [ ]:
df.to_csv('data/DNU_clasificadores_unificado.csv', index=False)
print(f"Guardado: data/DNU_clasificadores_unificado.csv  —  {df.shape[0]:,} filas x {df.shape[1]} columnas")